# 02 — Error Analysis and Evaluation

This notebook analyses saved evaluation outputs from the refactored repository.

Planned analyses:
- overall MAE / RMSE
- per-horizon MAE
- weighted MAE
- SSIM
- error concentration in high-intensity regions
- notes for uncertainty-aware extensions

In [1]:
from pathlib import Path

OUTPUTS_DIR = Path("../outputs")
METRICS_DIR = OUTPUTS_DIR / "metrics"
FIGURES_DIR = OUTPUTS_DIR / "figures"

print("Metrics dir exists:", METRICS_DIR.exists())
print("Figures dir exists:", FIGURES_DIR.exists())

Metrics dir exists: True
Figures dir exists: True


## Baseline evaluation context

The baseline checkpoint corresponds to the competition-aligned model:

- direct multi-output ConvLSTM-U-Net
- trained with MAE / L1 loss
- chosen to match the original competition scoring metric

In [2]:
metrics_path = METRICS_DIR / "baseline_test_metrics.json"
horizon_path = METRICS_DIR / "baseline_per_horizon_mae.csv"

print(metrics_path)
print(horizon_path)

../outputs/metrics/baseline_test_metrics.json
../outputs/metrics/baseline_per_horizon_mae.csv


## Overall metrics

This section will summarise:
- overall test MAE
- overall RMSE
- later: weighted MAE and SSIM

In [3]:
# Replace with actual loading once baseline evaluation has been run on desktop.
# Example:
# overall = pd.read_json(metrics_path, typ="series")
# overall

## Per-horizon error growth

One of the most important checks for this task is how error evolves across the 12-step forecast horizon.

A monotonic increase in MAE is expected:
- uncertainty compounds with lead time
- storm motion and growth/decay become harder to predict
- long-horizon frames are structurally more uncertain than near-horizon frames

The key question is whether the model remains stable and degrades gracefully.

In [4]:
# Replace with actual loading once metrics exist.
# horizon_df = pd.read_csv(horizon_path)
# ax = horizon_df.plot(x="horizon", y="mae", marker="o", figsize=(7, 4))
# ax.set_title("MAE per forecast horizon")
# ax.set_ylabel("MAE")
# plt.show()

## Why this evaluation matters

A single aggregate MAE can hide important behaviour.

For storm nowcasting, horizon-wise evaluation is essential because:
- short-horizon accuracy and long-horizon stability matter differently
- temporally unstable models can look acceptable on aggregate metrics
- visual drift and oversmoothing often emerge progressively with lead time

## Weighted MAE

The original model was trained with MAE because that was the competition metric.

Weighted MAE is introduced here as a **supplementary evaluation metric**, not a retroactive replacement of the benchmark.

Purpose:
- test whether high-intensity storm regions are under-emphasised by plain MAE
- quantify whether rare but operationally important pixels contribute disproportionately to errors

In [5]:
# Later:
# weighted_metrics_path = METRICS_DIR / "baseline_weighted_mae.json"
# pd.read_json(weighted_metrics_path, typ="series")

## SSIM

SSIM is also treated as a supplementary evaluation metric.

Why include it:
- MAE captures pixelwise error
- SSIM gives a structural fidelity signal
- storm forecasts can have reasonable MAE while still looking overly smooth or morphologically degraded

In [6]:
# Later:
# ssim_metrics_path = METRICS_DIR / "baseline_ssim.json"
# pd.read_json(ssim_metrics_path, typ="series")

## Expected failure modes

Based on the original notebook experiments and report:

- late-horizon frames are likely to show intensity smoothing
- rare high-intensity cores may be under-predicted
- pixelwise losses encourage conservative forecasts in ambiguous regions

These are exactly the motivations for adding weighted MAE and SSIM before changing the training objective.

## Planned extension: uncertainty proxy

A later extension will use MC dropout as a lightweight uncertainty proxy.

This will be framed carefully:
- not as calibrated probabilistic nowcasting
- not as operational confidence estimation
- but as a simple way to highlight where forecast uncertainty grows across space and horizon